# CASM end-to-end verification notebook

Walks the full pipeline against a baseline observation. Designed for the user to inspect before any merge of the `refactor-tower` branch.

Stages (in order): CAsMan refresh → load + diagnostics → fringe-stop → SVD calibrate → BF weights + inspect + transit + deploy (DADA only) → imaging → layout-version trace.

Live deploy to `corr1`/`corr2` is **explicitly commented out**. Uncomment only when you intend to push weights live.

Run from the integration venv:

```bash
source /home/casm/software/dev/casm_venvs/casm_refactor_env/bin/activate
jupyter lab notebooks/end_to_end.ipynb
```

## 1. Setup

In [ ]:
%matplotlib inline

import os
from pathlib import Path

# Proxy env vars are required for run_sync_wiring's GitHub fetch.
os.environ.setdefault("HTTP_PROXY",  "http://10.70.0.10:8118")
os.environ.setdefault("HTTPS_PROXY", "http://10.70.0.10:8118")

# Optional: pin the canonical layout if you want a specific one.
# os.environ["CASM_LAYOUT_CSV"] = "/home/casm/software/dev/antenna_layouts/casm_antenna_layout_may2026.csv"

# Imports across all five repos.
from casm_io.correlator import read_visibilities, AntennaMapping, load_format
from casm_io.constants import OVRO, C_LIGHT_M_PER_NS, resolve_layout_path

from casm_vis_analysis import (
    run_sync_wiring, run_build_layout,
    plot_autocorr, plot_waterfall, plot_fringe_diag, plot_phase_vs_freq,
    fringe_stop, FringeStoppedData,
    coherence_metric, auto_detect_sign,
    fit_delay, apply_delay,
    source_flux, RFIMask,
)
from casm_calibrator import (
    SVDCalibrator, SVDConfig, CalibrationResult, CalibrationWeightsWriter,
)
from bf_weights_generator import BFWeights, inspect_snap_weights
from casm_imaging import make_phased_sum_image

import matplotlib.pyplot as _plt
print('matplotlib backend:', _plt.get_backend())
print('Layout will resolve to:', resolve_layout_path())
print('OVRO site:', OVRO)
print('C / 1 ns =', C_LIGHT_M_PER_NS, 'm')

In [ ]:
import sys, subprocess
out = subprocess.run(
  [sys.executable, '-m', 'pip', 'install', '-e',
   '/home/casm/software/dev/.worktrees/refactor-tower/casm_vis_analysis',
   '--no-deps', '--force-reinstall'],
  capture_output=True, text=True,
)
print(out.stdout[-2000:])
print(out.stderr[-1000:])

## 2. CAsMan refresh (dry-run by default)

In [ ]:
# Pull a fresh CAsMan DB and show the wiring diff.
diff = run_sync_wiring(dry_run=True, force_pull=True)
print('added:',   len(diff['added']))
print('removed:', len(diff['removed']))
print('changed:', len(diff['changed']))

In [ ]:
# Build the consumer layout CSV (writes a dated file + updates the
# `current` symlink in $CASM_LAYOUT_DIR).
out = run_build_layout(dated=True, update_symlink=True, check_casman=True)
print(f"output: {out['output_csv']}  ({out['n_total']} rows, {out['n_active']} active)")
print(f"symlink updated: {out['symlink_updated']}")

## 3. Load + diagnostics

In [ ]:
# Pick the baseline observation here.
TIME_START = '2026-05-07 09:00:00'
TIME_END   = '2026-05-07 15:00:00'
TIME_TZ    = 'America/Los_Angeles'
DATA_ROOT  = '/mnt'
FORMAT     = '/home/casm/software/dev/casm_io/casm_io/correlator/configs/layout_64ant.json'

fmt = load_format(FORMAT)
ant = AntennaMapping.load()      # canonical resolver: $CASM_LAYOUT_CSV / current symlink
data = read_visibilities(
    time_start=TIME_START, time_end=TIME_END, time_tz=TIME_TZ,
    data_root=DATA_ROOT, fmt=fmt,
   # freq_range_mhz=(420.0, 450.0),    
)
print(f'vis:       {data.vis.shape}')
print(f'freq_mhz:  {data.freq_mhz.shape}  [{data.freq_mhz[0]:.1f} -> {data.freq_mhz[-1]:.1f}]')
print(f'time_unix: {data.time_unix.shape}')
print(f'antenna mapping: {ant}')

## 3.5  Apply RFI mask once (propagates downstream)

Set the per-channel RFI mask **once** here. Every downstream stage
(`plot_autocorr`, `plot_waterfall`, `fringe_stop`, `svd_calibrate`,
`save_calibration`) reads `data['freq_mask']` automatically — no
kwarg threading required.

What gets flagged is the OR of:
* **static** — the versioned `rfi_static_v1.json` shipped with
  `casm_vis_analysis` (currently flags amateur-satellite 437–438.5,
  LMR 451–452.5, GMRS/LMR 462.5–470, UHF TV ch14 474–485 MHz).
* **dynamic** — optional, populate later from auto-detection (SK,
  sigma-clip). Today this stays `None`.

Flagged channels appear as **white gaps** in plots, **NaN** in
`fs['vis_stopped']`, and are **excluded** from per-subband
averages in the SVD step. They are never interpolated at the
vis level — interpolation, when needed, happens at the gain
model in the calibrator (see § 5).

In [ ]:
from casm_vis_analysis import RFIMask, apply_rfi_mask

# Static mask: load the versioned config from the package.
static = RFIMask.from_static()                # latest version
# static = RFIMask.from_static(version=1)     # pin a specific version
print('static mask:', static.label, '|',
      len(static.bad_ranges_mhz), 'bands')

# Optional: a dynamic per-observation mask. Today stays None;
# wire SK / sigma-clip here when ready.
dynamic = None

apply_rfi_mask(data, static, dynamic=dynamic)
n_flagged = int(data['freq_mask'].sum())
print(f"freq_mask: {n_flagged}/{len(data['freq_mask'])} channels flagged "
      f"({100*n_flagged/len(data['freq_mask']):.1f}%)")

In [ ]:
ant = ant.with_inactive([2, 3, 12])


In [ ]:
# Compose API: one read, many plots.
auto_figs = plot_autocorr(data, ant, scale='dB', ncols=3)


In [ ]:
wf_figs   = plot_waterfall(data, ant, split_max=len(ant.active_antennas()))

## 4. Fringe-stop

In [ ]:

# here so the cell runs end-to-end without manual baseline indexing.
fs = fringe_stop(data, ant, ref_ant=9, source='sun', sign=-1)
print('fs keys:', list(fs.keys()))
print('targets:', fs['target_aids'])
print('vis_stopped shape:', fs['vis_stopped'].shape)
print('fs keys:', list(fs.keys()))

In [ ]:
from casm_vis_analysis import fit_delay, apply_delay, plot_fringe_diag, plot_phase_vs_freq
import numpy as np
# Linear delay fit on top of fringe-stopped vis (diagnostic, NOT calibration input).
# Average only over the transit window — full-window averaging dilutes raw
# magnitude with non-source samples and turns raw phase into noise.
time_mask = fs['time_mask']
delay_params = fit_delay(
    fs['vis_stopped'], fs['freq_mhz'],
    time_mask=time_mask,
    freq_mask=fs['freq_mask'],   # exclude RFI from the linear fit
    model='linear',
)
vis_delay_corrected = apply_delay(fs['vis_stopped'], fs['freq_mhz'],
                                delay_params, model='linear')

# 4-panel diagnostic
panels = [
  ("Raw phase",            fs['vis']),
  ("Geometric",            fs['geometric_phase']),
  ("Fringe-stopped",       fs['vis_stopped']),
  ("Post-delay (linear)",  vis_delay_corrected),
]
ref_snap, _ = ant.snap_adc(fs['ref_ant'])
target_snaps = [ant.snap_adc(aid)[0] for aid in fs['target_aids']]

diag_figs = plot_fringe_diag(
  panels, fs['time_unix'], fs['freq_mhz'],
  target_labels=fs['target_labels'],
  target_snaps=target_snaps,
  ref_snap=ref_snap,
  freq_mask=fs['freq_mask'],   # white rows at flagged channels
  output_dir=None,
)

# Phase vs freq grid: one column per stage, one row per baseline.
# Each panel has independent y-axis (raw unwrap swings ±100s of rad,
# FS / post-delay sit within a few rad — single shared axis would
# collapse the corrected stages to a flat line).
phase_panels = [(lbl, v) for lbl, v in panels if lbl != "Geometric"]
phase_fig = plot_phase_vs_freq(
  phase_panels, fs['freq_mhz'],
  baseline_labels=fs['target_labels'],
  freq_mask=fs['freq_mask'],   # gaps at RFI; clean unwrap on good channels only
  time_mask=time_mask,         # transit window, matches the linear-delay fit
  time_unix=fs['time_unix'],
  output_path=None,
)


In [ ]:
phase_panels = [(lbl, v) for lbl, v in panels if lbl != "Geometric"]
phase_fig = plot_phase_vs_freq(
  phase_panels, fs['freq_mhz'],
  baseline_labels=fs['target_labels'],
  freq_mask=fs['freq_mask'],   # gaps at RFI; clean unwrap on good channels only
  time_mask=time_mask,         # transit window, matches the linear-delay fit
  time_unix=fs['time_unix'],
  output_path=None,
)

## 4.1  Delay-fit diagnostics

Visual confirmation of the linear-delay fits, plus a physical sanity check
(delay vs antenna baseline length).

The fit itself is rerun with `return_coherence=True` so the coherence curve
`C(τ) = |Σ_f vis(f)·exp(-i·2π·τ·f)|` is available for plotting. That curve
is what the algorithm actually maximises — its shape tells you how
confident the fit is for each baseline.


In [ ]:
# Re-fit with coherence curves stored, then read every diagnostic field.
from casm_vis_analysis import (
    fit_delay, plot_coherence_curves, plot_delay_vs_baseline_length,
    compute_baselines_enu,
)

delay_params = fit_delay(
    fs['vis_stopped'], fs['freq_mhz'],
    time_mask=fs['time_mask'],
    freq_mask=fs['freq_mask'],
    model='linear',
    return_coherence=True,        # adds 'coherence_curves' + 'tau_grid_ns'
)

print(f"τ_Nyquist (Δν-limited): {delay_params['tau_nyquist_ns']:.0f} ns")
print(f"{'idx':>3}  {'baseline':<28}  {'τ (ns)':>9}  {'r²':>6}  "
      f"{'pk/2nd':>7}  flag")
print("-" * 72)
for i, lbl in enumerate(fs['target_labels']):
    flag = 'LOW QUALITY' if delay_params['low_quality'][i] else ''
    print(f"{i:>3}  {lbl:<28}  "
          f"{delay_params['delay_ns'][i]:>9.2f}  "
          f"{delay_params['r_squared'][i]:>6.3f}  "
          f"{delay_params['peak_to_secondary_ratio'][i]:>7.2f}  {flag}")


In [ ]:
# Per-baseline coherence curves. Red border = low_quality flag.
# Each panel: |C(τ)| / max, with the chosen delay marked in red.
coh_figs = plot_coherence_curves(
    delay_params,
    target_labels=fs['target_labels'],
    output_path=None,             # set a path string to save instead
    split_max=20,
)


In [ ]:
# Delay vs antenna baseline length, with twin axis showing the
# equivalent cable-length difference at the coax velocity factor.
# 1 ns of delay  ↔  v_f · c · 1 ns ≈ 20 cm of cable at v_f = 0.67.
# Points are coloured by target SNAP so per-chassis cable clusters stand out.
import numpy as np

positions   = ant.get_positions()                      # (n_rows, 3) ENU
aid_to_row  = {int(a): i for i, a in enumerate(ant.dataframe['antenna_id'].values)}
ref_idx     = aid_to_row[fs['ref_ant']]
target_idxs = [aid_to_row[a] for a in fs['target_aids']]
baselines   = compute_baselines_enu(positions, ref_idx, target_idxs)
baseline_len_m = np.linalg.norm(baselines, axis=1)

target_snaps = [ant.snap_adc(a)[0] for a in fs['target_aids']]

fig = plot_delay_vs_baseline_length(
    delay_params['delay_ns'],
    baseline_len_m,
    target_labels=fs['target_labels'],
    target_snaps=target_snaps,    # colour by SNAP, with legend
    low_quality=delay_params['low_quality'],
    velocity_factor=0.67,         # coax
    output_path=None,
)


In [ ]:
fs.keys()

## 5. SVD calibrate

The compose-style `svd_calibrate(fs, ant)` is staged but the full dict-wrapper lands with the VisibilityMatrix dedup. For the verification run, use `SVDCalibrator` directly with a `VisibilityMatrix` you build yourself (legacy path).

In [ ]:
from casm_calibrator import svd_calibrate, SVDConfig, SVDMode
cfg = SVDConfig(
      threshold=2.0,
      svd_mode=SVDMode.PHASE_ONLY,
      block_size=32,
  )
cal = svd_calibrate(fs, ant, data=data, config=cfg)

print(f"gains:  {cal['gains'].shape}   ({cal['gains'].dtype})")
print(f"good ch: {int(cal['flags'].sum())} / {len(cal['flags'])}")
print(f"lambda1/lambda2 (passing): "
    f"{cal['rank1_ratios'][cal['flags']].min():.2f} – "
    f"{cal['rank1_ratios'][cal['flags']].max():.2f}")
print(f"ref_ant_id: {cal['ref_ant_id']}    source: {cal['source']}")

In [ ]:
import numpy as np
g = fs['freq_mask']
tm = time_mask
print('time_mask True count:', int(tm.sum()), '/', len(tm))
print('freq_mask True count:', int(g.sum()), '/', len(g))

for label, arr in [
  ('Raw',           fs['vis']),
  ('FS',            fs['vis_stopped']),
  ('PD',            vis_delay_corrected),
]:
  finite_frac = np.isfinite(arr).mean()
  sliced = arr[tm][:, :, 0]
  avg = np.mean(sliced, axis=0)
  mag = np.abs(avg)
  phase_good = np.angle(avg[g])
  print(f'{label:5s}  finite={finite_frac:.4f}'
        f'  |mean|@good min/median/max = '
        f'{mag[g].min():.3e} / {np.median(mag[g]):.3e} / {mag[g].max():.3e}'
        f'  phase[good] finite={np.isfinite(phase_good).all()}')

### Subband SVD with smooth gain interpolation (RFI-aware)

Alternative SVD config: skip RFI-flagged channels inside each
subband mean, fail the subband if too few clean channels remain or
if λ₁/λ₂ is below threshold, then fit a smooth polynomial across
all valid subbands per antenna.

Same `cal` dict as before — drops into `plot_calibration`,
`save_calibration`, `generate_combined_weights` unchanged.

In [ ]:
from casm_calibrator import svd_calibrate, SVDConfig, SVDMode

cfg_sb = SVDConfig(
    threshold=1.5,           # per-subband lambda_1/lambda_2 floor
    svd_mode=SVDMode.PHASE_ONLY,
    subband_size=8,         # channels per subband
    subband_min_good_frac=0.5,
    smooth_order=3,          # polynomial order for the per-antenna fit
    fill_failed='smooth_fit',  # or 'zero'
    # Optional: drop short baselines (in wavelengths at the highest in-band
    # frequency) to remove large-scale extended emission. None = keep all
    # baselines (default). Standard self-cal practice for unresolved sources
    # like Cyg A is 1.5; for the resolved Sun, empirically leave at None
    # (cutting shorts removes signal more than contamination).
    min_baseline_wavelengths=None,
)
cal = svd_calibrate(fs, ant, data=data, config=cfg_sb)

n_subbands = cal['rank1_ratios'].shape[0] // cfg_sb.subband_size
print(f"channels:  {len(cal['flags'])}")
print(f"good ch:   {int(cal['flags'].sum())}")
print(f"lambda1/lambda2 median (good ch): "
      f"{float(np.median(cal['rank1_ratios'][cal['flags']])):.2f}")


In [ ]:
import numpy as np

# Narrow the average to ±30 min around transit
times = fs['time_unix']
t_center = times[fs['time_mask']].mean()  # transit center within current window
narrow = (np.abs(times - t_center) < 1800)   # ±30 min
print(f"Narrow mask: {narrow.sum()}/{len(narrow)} samples")

cal_narrow = svd_calibrate(fs, ant, data=data,
                         config=cfg_sb, time_mask=narrow)

# Use the same per-subband summary the comparison did:
r = cal_narrow['rank1_ratios']
sb = cfg_sb.subband_size
sb_ratios = np.array([r[i*sb:(i+1)*sb].mean() for i in range(len(r)//sb)])
sb_ratios = sb_ratios[np.isfinite(sb_ratios) & (sb_ratios > 0)]
print(f"Narrow-window median λ₁/λ₂: {np.median(sb_ratios):.2f}  "
    f"(was 2.52 with full 6h window)")


In [ ]:
# Diagnostic plots for the SVD calibration.
# `plot_calibration` returns 3 panels (rank1, wrapped phase, amplitude),
# but only rank1 is useful as-is: amplitude is flat 1.0 in PHASE_ONLY mode,
# and the wrapped phase plot is unreadable across phase wraps. We inline
# rank1 + an UNWRAPPED phase plot here.
import numpy as np
import matplotlib.pyplot as plt

flags     = np.asarray(cal['flags'])
ratios    = np.asarray(cal['rank1_ratios'])
gains     = cal['gains']
freqs     = np.asarray(cal['freqs_mhz'])
ant_ids   = list(cal['ant_ids'])
n_ant     = gains.shape[0]
threshold = cfg_sb.threshold

def _ant_label(aid):
    base = f'Ant {aid}'
    try:
        row = ant.dataframe.loc[ant.dataframe.antenna_id == aid].iloc[0]
        grid = f' {row["row"]}{row["col"]}' if row.get('row') and row.get('col') else ''
        return f'{base}{grid}'
    except Exception:
        return base

# --- (1) Rank-1 quality vs frequency ---
fig1, ax = plt.subplots(figsize=(13, 4))
clip = np.clip(ratios, 0, 50)
ax.scatter(freqs, clip, c=np.where(flags, 'C0', 'red'), s=4, alpha=0.6)
ax.axhline(threshold, color='orange', ls='--', lw=1.2, label=f'threshold = {threshold:.1f}')
ax.set_xlabel('Frequency (MHz)')
ax.set_ylabel(r'$\lambda_1 / \lambda_2$')
ax.set_title(f"Rank-1 quality ({cal.get('source','')}) — "
             f"{int(flags.sum())}/{len(flags)} channels passed")
ax.set_xlim(freqs[0], freqs[-1])
ax.set_ylim(0, max(threshold * 1.5, clip.max() * 1.05))
ax.legend(loc='upper right')
fig1.tight_layout()

# --- (2) Per-antenna gain phase, unwrapped per contiguous good-channel run ---
# Unwrapping the whole array would inject 2pi jumps across RFI gaps; instead
# we unwrap each contiguous good-channel run separately so each antenna's
# cable-delay ramp is a clean continuous line within each cleanband.
fig2, ax = plt.subplots(figsize=(13, 4))
for k in range(n_ant):
    phase_rad = np.angle(gains[k]).astype(float)
    phase_rad[~flags] = np.nan
    if not np.isfinite(phase_rad).any():
        continue
    unwrapped = np.full_like(phase_rad, np.nan)
    good_idx = np.where(np.isfinite(phase_rad))[0]
    breaks = np.where(np.diff(good_idx) > 1)[0]
    starts = np.r_[0, breaks + 1]
    ends   = np.r_[breaks + 1, len(good_idx)]
    for s, e in zip(starts, ends):
        seg = good_idx[s:e]
        unwrapped[seg] = np.unwrap(phase_rad[seg])
    ax.plot(freqs, np.degrees(unwrapped), lw=0.8, alpha=0.85, label=_ant_label(ant_ids[k]))
ax.set_xlabel('Frequency (MHz)')
ax.set_ylabel('Gain phase (deg, unwrapped per cleanband)')
ax.set_title(f"Per-antenna gain phase ({cal.get('source','')}) — unwrapped")
ax.set_xlim(freqs[0], freqs[-1])
ax.legend(fontsize=7, ncol=max(1, n_ant // 8),
          loc='center left', bbox_to_anchor=(1.0, 0.5))
fig2.tight_layout()
plt.show()


## 6. Beamforming weights (full recipe up to deploy)

Builds on `cal` from the SVD section above. Steps:

1. Save cal as HDF5 (the format `bf_weights_generator` consumes).
2. Pick the beam grid (3 idioms shown).
3. Combine geometric + cal weights.
4. Save complex64 + int8 HDF5 files.
5. Inspect what landed.
6. Plot source transits across the beams.
7. Stage DADA files for deploy (live-upload cell stays commented out).

In [ ]:
import inspect
print('source file:', inspect.getsourcefile(plot_phase_vs_freq))
print('signature:  ', inspect.signature(plot_phase_vs_freq))
print('axes count: ', len(phase_fig.axes))
print('first row of fig.axes:', phase_fig.axes[:5])

In [ ]:
# 6.1  Save the SVD calibration as HDF5 (the form bf_weights_generator wants).
from casm_calibrator import save_calibration

CAL_H5 = '/tmp/cal_demo.h5'
save_calibration(cal, CAL_H5, n_time_averaged=fs['vis_stopped'].shape[0])
print('wrote', CAL_H5)

In [ ]:
# 6.2  Pick a beam grid. Three idioms — uncomment the one you want.
from bf_weights_generator import (
    StationaryPointing,
    generate_beam_grid_altaz,    # rectangular alt/az grid
    generate_beam_grid_lm,       # uniform direction-cosine grid
    generate_beam_grid,          # target n_beams + array-aware FWHM
    Array64Config,
)

# (a) Explicit pointings (e.g. transit at known sources):
# beams = [
#     StationaryPointing.from_altaz(60.0, 180.0, name='sun_60S'),
#     StationaryPointing.from_altaz(45.0,  90.0, name='ne_45'),
# ]

# (b) Uniform alt/az grid:
# beams = generate_beam_grid_altaz(
#     alt_min_deg=30.0, alt_max_deg=85.0,
#     az_min_deg=0.0,   az_max_deg=360.0,
#     spacing_deg=10.0, exclude_horizon=True,
# )

# (c) Direction-cosine grid (uniform sky coverage):
# beams = generate_beam_grid_lm(spacing=0.07)

# (d) Target a beam count + auto-spacing from the array geometry:
arr = Array64Config.from_antenna_mapping(ant)   # uses in-memory AntennaMapping (honours with_inactive)
beams = generate_beam_grid(
    n_beams=64,
    alt_min_deg=30.0, alt_max_deg=90.0,
    array_config=arr,
    freq_hz=437.5e6,
)
print(f'{len(beams)} beams')

In [ ]:
# 6.3  Combine geometry + cal -> complex64 weights ready for the F-engine.
from bf_weights_generator import (
    load_calibration_weights, generate_combined_weights,
    save_combined_weights_hdf5, save_int8_weights_hdf5,
)

cal_weights = load_calibration_weights(CAL_H5)

combined = generate_combined_weights(
    pointing=beams,
    array_config=arr,
    cal_weights=cal_weights,
    freq_order='descending',
)
print('combined.weights shape:', combined.weights.shape,
      '  (n_beams, 64 SNAP slots, n_chan)')

In [ ]:
# 6.4  Save: complex64 (full precision) + int8 (deployment quantization).
# CombinedWeights holds complex64; quantize via .to_int8() to get the
# F-engine wire format that save_int8_weights_hdf5 expects.
WEIGHTS_H5      = '/tmp/my_weights.h5'
WEIGHTS_INT8_H5 = '/tmp/my_weights_int8.h5'
save_combined_weights_hdf5(combined,             WEIGHTS_H5,      overwrite=True)
save_int8_weights_hdf5    (combined.to_int8(),   WEIGHTS_INT8_H5, overwrite=True)
print('wrote', WEIGHTS_H5)
print('wrote', WEIGHTS_INT8_H5)


In [ ]:
# 6.5  Sanity-check the int8 file: alt/az coverage, beam count,
#      which SNAP inputs carry non-zero weights.
from bf_weights_generator import inspect_snap_weights

info = inspect_snap_weights(WEIGHTS_INT8_H5,
                            layout=str(out['output_csv']))
print()
print(f"#beams: {info['n_beams']}")
print(f"alt range: {min(info['beam_alt_deg']):.1f}° – "
      f"{max(info['beam_alt_deg']):.1f}°")
print(f"az  range: {min(info['beam_az_deg']):.1f}° – "
      f"{max(info['beam_az_deg']):.1f}°")
print(f"non-zero SNAP inputs: {info['n_nonzero_snap_inputs']} / 64")

In [ ]:
# 6.6  Where do bright sources transit through these beams today?
from bf_weights_generator import plot_source_transit

fig = plot_source_transit(
    WEIGHTS_INT8_H5,
    sources=['sun', 'cas-a', 'cyg-a'],
    date='2026-05-07',                   # YYYY-MM-DD; or omit for today
    time_tz='America/Los_Angeles',
    time_start='06:00', time_end='15:00',
    output_path=None,                    # render inline
)

In [ ]:
# 6.7  Stage DADA files (the format the SNAP F-engine consumes).
#      `deploy_bf_weights` is uncommitted on production main, so for
#      now invoke the script form. It writes direct.dada.{0..5} into
#      the cwd (or --output-dir).
import subprocess, sys, os
DEPLOY_OUT = '/tmp/snap_outputs'
os.makedirs(DEPLOY_OUT, exist_ok=True)
subprocess.run([
    sys.executable,
    '/home/casm/software/dev/bf_weights_generator/bf_weights_generator/deploy_bf_weights.py',
    WEIGHTS_INT8_H5,
    '--output-dir', DEPLOY_OUT,
    '--dry-run',
], check=False)
print('staged DADA files in', DEPLOY_OUT)


In [ ]:
# ----- LIVE DEPLOY: DO NOT RUN unless you intend to push to corr1/corr2 -----
#
# subprocess.run([
#     sys.executable,
#     '/home/casm/software/dev/bf_weights_generator/deploy_bf_weights.py',
#     WEIGHTS_INT8_H5,
#     '--upload',
#     '--save-defaults',
#     '--utc-start', '2026-MM-DD-HH:MM:SS',
# ], check=True)

## 6.8  Calibration validation: beam power vs time toward known sources

Apply the SVD-derived per-antenna gains to the raw visibilities and beamform towards Sun, Cas A, Cyg A, Tau A, plus a fixed off-source control. If the calibration phases the array up correctly, the calibrator (Sun, here) should dominate while it's up; the other beams should sit near the noise floor or pick up sidelobes only.

This is the cheapest test of "did calibration work" before the B0329+54 pulsar fold.

In [ ]:
from casm_vis_analysis import beam_power_vs_time, plot_beam_power
from bf_weights_generator import CalibrationWeights
import numpy as np

# Repackage the SVD output as a CalibrationWeights container so beam_power_vs_time
# can apply per-antenna gains to the raw visibilities.
cal_w = CalibrationWeights(
    weights=cal['weights'].astype(np.complex64),
    flags=cal['flags'],
    frequencies_hz=(cal['freqs_mhz'] * 1e6).astype(np.float64),
    ant_ids=np.asarray(cal['ant_ids']),
    ref_ant_id=int(cal['ref_ant_id']),
    source=cal.get('source', 'sun'),
)

result = beam_power_vs_time(
    data, ant,
    sources=[
        'sun', 'cas-a', 'cyg-a', 'tau-a',
        ('zenith',           90.0,   0.0),
        ('south_30deg_alt',  30.0, 180.0),
    ],
    cal_weights=cal_w,
    freq_band_mhz=(405, 433),     # cleanband below the v2 RFI region
)
for label, p in result['power'].items():
    alt_pk = result['alt_deg'][label].max()
    print(f"  {label:14s}  median={np.median(p):>+.3e}  max={p.max():>+.3e}  alt_peak={alt_pk:.1f}°")

fig = plot_beam_power(result)


## 7. Imaging

In [ ]:
# CLI mirror: `casm-bf-image --source sun --start ... --end ...`
img = make_phased_sum_image(
    source='sun',
    time_start=TIME_START,
    time_end=TIME_END,
    output_dir='./images',
    ang_max_deg=15.0,
    npix=101,
    fs_sign=-1,
    fast_mode=True,
    compute_snr=True,
    verbose=True,
)
if img.get('snr_info') is not None:
    print('Measured SNR:', img['snr_info']['snr'])

## 8. Layout-version trace

In [ ]:
# Confirm every artifact references the same canonical CSV.
print('canonical layout (resolved):', resolve_layout_path())
print('build_layout output_csv:    ', out['output_csv'])
# When cal NPZ + weights HDF5 are produced, also print their layout_version stamps:
# import numpy as np
# cal = np.load('cal_demo.npz', allow_pickle=True)
# print('cal layout_version:', dict(cal['layout_version'].item()))

## 9. Per-beam validation against deployed weights

Before deploying weights to the F-engine, confirm the int8 weights file we just wrote actually steers the array on the sky. The flow:

1. Read the 64 beam pointings from `WEIGHTS_INT8_H5`.
2. Predict which (beam, source) pairs the configured sources cross during the data window (geometric, FWHM-based).
3. For each beam with a predicted hit (and a couple of no-hit controls), beamform the in-memory visibilities at that exact alt/az using the cal weights — see `casm_vis_analysis.beam_power_vs_time` for the underlying math.
4. Compare peak power inside the predicted transit window vs. median power outside. Pass = ratio ≥ 5.

If a beam pointed at the Sun's transit position picks up a clear power peak inside the predicted window, the cal/geo combination is producing real beams — the weights are safe to deploy.

CLI mirror:
```bash
casm-validate-bf-weights /tmp/my_weights_int8.h5 \
    --cal-h5 /tmp/cal_demo.h5 \
    --time-start "2026-05-08 11:30" --time-end "2026-05-08 14:30" \
    --time-tz America/Los_Angeles \
    --inactive 2 3 12 \
    -o /tmp/bf_validation.png
```

In [ ]:
from casm_vis_analysis import validate_beam_weights, plot_beam_validation

result = validate_beam_weights(
    int8_h5=WEIGHTS_INT8_H5,
    data=data, ant=ant, cal_weights=cal_w,
    sources=('sun', 'cas-a', 'cyg-a', 'tau-a'),
    freq_band_mhz=(405, 433),
    max_hit_panels=8,
    n_control_beams=2,
)
n_pass = sum(1 for m in result['per_beam_metrics'].values()
             if m.get('expected_hit') and m['pass'])
n_total = sum(1 for m in result['per_beam_metrics'].values()
              if m.get('expected_hit'))
print(f'Beams with predicted source transit: {n_total}, passing ratio>=5: {n_pass}')
for bi, m in sorted(result['per_beam_metrics'].items()):
    if m.get('expected_hit'):
        tag = 'PASS' if m['pass'] else 'FAIL'
        print(f"  [{tag}] beam {bi:3d}  ratio={m['ratio']:6.2f}  sources={m['sources']}")
    else:
        print(f"  [ctrl] beam {bi:3d}  peak_abs={m['peak_abs']:.2e}")

fig = plot_beam_validation(result)
